<a href="https://colab.research.google.com/github/ankit-kothari/Data-Science-Journey/blob/master/encoder_only_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install dask
!pip install 'fsspec>=0.3.3'
!pip install datasets
!pip install torchinfo


In [ ]:
!pip3 install BertViz

In [ ]:
from transformers import AutoTokenizer
from bertviz.transformers_neuron_view import BertModel
from bertviz.neuron_view import show

# Encoder
- Encoder consity of stacks of Enoder Layers. Each Layer has
  - MultiHead Self Attention
  - A fully connected Feed Forward layer that is applied to each input Embeddings

## Part 1: Multi-Head Self Attention Building Block
- calculating attention at every Head(12 in this example)

#### Tokens

In [ ]:
model_ckpt = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = BertModel.from_pretrained(model_ckpt)
text = "time flies like an arrow"

Downloading:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/570 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/226k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/455k [00:00<?, ?B/s]

100%|██████████| 440473133/440473133 [00:16<00:00, 27398256.19B/s]


In [ ]:
show(model,"bert", tokenizer, text, display_mode='light', layer=0, head=8)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#### Tokenization (Text to Integers)

In [ ]:
inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False)
inputs.input_ids

tensor([[ 2051, 10029,  2066,  2019,  8612]])

In [ ]:
from torch import nn
from transformers import AutoConfig

In [ ]:
config = AutoConfig.from_pretrained(model_ckpt)

In [ ]:
config

BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.20.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

#### Token Embeddings (Intergers to Dense Representation)

In [ ]:
token_embedding = nn.Embedding(config.vocab_size, config.hidden_size)

In [ ]:
token_embedding

Embedding(30522, 768)

In [ ]:
input_embedding = token_embedding(inputs.input_ids)
input_embedding

tensor([[[-0.8211, -1.3846, -0.8913,  ...,  0.5060,  1.7496, -1.0207],
         [-0.8271,  1.9188, -0.8753,  ..., -1.0678,  1.4826, -1.0137],
         [-0.2271, -2.3658, -1.1294,  ...,  0.1230, -0.8667,  0.6972],
         [-0.5537,  1.3866,  1.0338,  ...,  1.2735, -1.9913, -1.9515],
         [-0.3893,  0.8792, -0.5036,  ...,  0.8621,  1.0775,  0.4856]]],
       grad_fn=<EmbeddingBackward0>)

In [ ]:
# batch_size * seq_length * hidden_dimension
input_embedding.shape

torch.Size([1, 5, 768])

#### Creatings query (q) , Key (k) and Value (v) Vector Matrices

In [ ]:
query = key = value = input_embedding

In [ ]:
query.shape

torch.Size([1, 5, 768])

In [ ]:
dim_k = key.size(-1)
dim_k

768

In [ ]:
key.transpose(1,2).shape

torch.Size([1, 768, 5])

#### Multiplying Q  X K.T (Calculating attention score)
- Calculate attention for each word with eevery other word
- q1.k1 + q1.k2 + q1.k3 + ......q1.kn
- Divided by the square_root of embedding dimension

In [ ]:
import torch
from math import sqrt
scores = torch.bmm(query, key.transpose(1,2)) / sqrt(dim_k)

In [ ]:
scores.shape

torch.Size([1, 5, 5])

#### Calculating Softmax

In [ ]:
import torch.nn.functional as F

In [ ]:
scores

tensor([[[30.3602,  0.7506,  0.4277, -1.7896,  0.9505],
         [ 0.7506, 26.7841,  1.3366, -0.2221, -0.4218],
         [ 0.4277,  1.3366, 28.3035,  0.6716,  0.0433],
         [-1.7896, -0.2221,  0.6716, 27.5665, -0.4464],
         [ 0.9505, -0.4218,  0.0433, -0.4464, 28.5416]]],
       grad_fn=<DivBackward0>)

In [ ]:
weights = F.softmax(scores, dim=-1)

In [ ]:
weights.shape

torch.Size([1, 5, 5])

In [ ]:
weights[0,1,:]

tensor([4.9407e-12, 1.0000e+00, 8.8771e-12, 1.8678e-12, 1.5297e-12],
       grad_fn=<SliceBackward0>)

In [ ]:
sum(weights[0,1,:])

tensor(1., grad_fn=<AddBackward0>)

In [ ]:
weights.sum(dim=-1)

tensor([[1., 1., 1., 1., 1.]], grad_fn=<SumBackward1>)

#### Calculating Self Attention for each Head

In [ ]:
#multiply the attention weights with the values
attn_outputs = torch.bmm(weights,value)           #(1,5,5) * (1,5,768) ----> (1,5,768)

In [ ]:
attn_outputs.shape

torch.Size([1, 5, 768])

In [ ]:
def scaled_dot_product_attention(query, key, value):
  dim_k = query.size(dim=-1)
  print(f'Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head {dim_k}')
  scores = torch.bmm(query,key.transpose(1,2))/ sqrt(dim_k)  #[(1,5,768)*(1,768,5)]/sqrt(768) >>> [batch_size,5,5]
  print(f'Scores are calculated by multiplying q dot k.T {scores.shape}')
  weights = F.softmax(scores, dim=-1) #[batch_size,5,5]
  print(f'Softmax for each column across one row {weights.shape}')
  weights_dot_values = torch.bmm(weights,value)
  print(f'Last Step is to multiply weights and values {weights_dot_values.shape}')
  return weights_dot_values

In [ ]:
from torch import nn
class AttentionHead(nn.Module):
  def __init__(self, embed_dim, head_dim):
    super().__init__()
    self.head_dim = head_dim
    #infeatures=embed_dim
    #outfeatures=head_dim
    self.q = nn.Linear(embed_dim, head_dim)
    self.k = nn.Linear(embed_dim, head_dim)
    self.v = nn.Linear(embed_dim, head_dim)

  def forward(self, hidden_state):
    print(f'Input Embedding for Each Token with X Matrix {hidden_state.size()}')
    #q = X*W_q
    q = self.q(hidden_state)
    print(f'Shape of the Query Matrix W_q {q.size()}')
    k = self.k(hidden_state)
    print(f'Shape of the Key Matrix W_k {k.size()}')
    v = self.k(hidden_state)
    print(f'Shape of the Value Matrix W_k {v.size()}')
    print('-----------------Calculating Self Attention--------------------')
    attn_outputs = scaled_dot_product_attention(q,k,v)
    print(f'Shape of the attention Output with one Head and Head Dimension {self.head_dim} is {attn_outputs.size()}')
    return attn_outputs

In [ ]:
head_1 = AttentionHead(768,64)

In [ ]:
head_1

AttentionHead(
  (q): Linear(in_features=768, out_features=64, bias=True)
  (k): Linear(in_features=768, out_features=64, bias=True)
  (v): Linear(in_features=768, out_features=64, bias=True)
)

In [ ]:
input_embedding.shape

torch.Size([1, 5, 768])

In [ ]:
attn_outputs_1 = head_1(input_embedding) #infeatures= embed_dim---> making of the X matrix

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
attn_outputs_1.size()

torch.Size([1, 5, 64])

In [ ]:
head_2 = AttentionHead(768,64)
attn_outputs_2 = head_2(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_3 =  AttentionHead(768,64)
attn_outputs_3 = head_3(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_4 =  AttentionHead(768,64)
attn_outputs_4 = head_4(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_5 =  AttentionHead(768,64)
attn_outputs_5 = head_5(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_6 =  AttentionHead(768,64)
attn_outputs_6 = head_6(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_7 =  AttentionHead(768,64)
attn_outputs_7 = head_7(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_7 =  AttentionHead(768,64)
attn_outputs_7 = head_7(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_9 =  AttentionHead(768,64)
attn_outputs_9 = head_9(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_10 =  AttentionHead(768,64)
attn_outputs_10 = head_10(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_11 =  AttentionHead(768,64)
attn_outputs_11 = head_11(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


In [ ]:
head_12 =  AttentionHead(768,64)
attn_outputs_12 = head_12(input_embedding)

Input Embedding for Each Token with X Matrix torch.Size([1, 5, 768])
Shape of the Query Matrix W_q torch.Size([1, 5, 64])
Shape of the Key Matrix W_k torch.Size([1, 5, 64])
Shape of the Value Matrix W_k torch.Size([1, 5, 64])
-----------------Calculating Self Attention--------------------
Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head 64
Scores are calculated by multiplying q dot k.T torch.Size([1, 5, 5])
Softmax for each column across one row torch.Size([1, 5, 5])
Last Step is to multiply weights and values torch.Size([1, 5, 64])
Shape of the attention Output with one Head and Head Dimension 64 is torch.Size([1, 5, 64])


#### Concating Attention Outputs of all the Heads (12)
z0+ z1 + z2 + ........z12

In [ ]:
concat = torch.cat([attn_outputs_1,attn_outputs_2,attn_outputs_3,attn_outputs_4,attn_outputs_5,attn_outputs_6,
                    attn_outputs_7,attn_outputs_8,attn_outputs_9,attn_outputs_10,attn_outputs_11,attn_outputs_12], dim=-1)

In [ ]:
concat.size()

torch.Size([1, 5, 768])

#### Creating the Weight Matrix W_0

In [ ]:
embed_dim = 768
w_0_matrix = nn.Linear(embed_dim,embed_dim)
w_0_matrix

Linear(in_features=768, out_features=768, bias=True)

#### Final Output

- Same shape as the input embedding [batch_size, seq_len, hidden_state_dim]

In [ ]:
Z = w_0_matrix(concat)

In [ ]:
Z.shape

torch.Size([1, 5, 768])

#### Creating MultiHeadAttention class

In [ ]:

def scaled_dot_product_attention(query, key, value):
  dim_k = query.size(dim=-1)
  #print(f'Dimension of the q,k,v Matrix [Batch_size, seq_len, Head_dim] of One Head {dim_k}')
  scores = torch.bmm(query,key.transpose(1,2))/ sqrt(dim_k)  #[(1,5,768)*(1,768,5)]/sqrt(768) >>> [batch_size,5,5]
  #print(f'Scores are calculated by multiplying q dot k.T {scores.shape}')
  weights = F.softmax(scores, dim=-1) #[batch_size,5,5]
  #print(f'Softmax for each column across one row {weights.shape}')
  weights_dot_values = torch.bmm(weights,value)
  #print(f'Last Step is to multiply weights and values {weights_dot_values.shape}')
  return weights_dot_values

from torch import nn
class AttentionHead(nn.Module):
  def __init__(self, embed_dim, head_dim):
    super().__init__()
    self.head_dim = head_dim
    #infeatures=embed_dim
    #outfeatures=head_dim
    self.q = nn.Linear(embed_dim, head_dim)
    self.k = nn.Linear(embed_dim, head_dim)
    self.v = nn.Linear(embed_dim, head_dim)

  def forward(self, hidden_state):
    #print(f'Input Embedding for Each Token with X Matrix {hidden_state.size()}')
    #q = X*W_q
    q = self.q(hidden_state)
    #print(f'Shape of the Query Matrix W_q {q.size()}')
    k = self.k(hidden_state)
    #print(f'Shape of the Key Matrix W_k {k.size()}')
    v = self.k(hidden_state)
    #print(f'Shape of the Value Matrix W_k {v.size()}')
    #print('-----------------Calculating Self Attention--------------------')
    attn_outputs = scaled_dot_product_attention(q,k,v)
    #print(f'Shape of the attention Output with one Head and Head Dimension {self.head_dim} is {attn_outputs.size()}')
    return attn_outputs

class MultiHeadAttention(nn.Module):
  def __init__(self,config):
    super().__init__()
    embed_dim = config.hidden_size
    num_heads = config.num_attention_heads
    head_dim = embed_dim // num_heads
    self.heads = [AttentionHead(embed_dim, head_dim) for _ in range(num_heads)]
    self.w_0 = nn.Linear(embed_dim,embed_dim)

  def forward(self,hidden_state):
    '''
    hidden_state: Input Embedding with dimensions [batch_size, seq_len, embedding_dimension]
    '''
    attention_outputs = [head(hidden_state) for head in self.heads] #Calculating Self-Attention on each head
    contcat_attn_outputs_allheads = torch.cat(attention_outputs, dim=-1) #[batch_size,seq_len, embed_dim]
    Z =   self.w_0(contcat_attn_outputs_allheads) #[batch_size, seq_len, embed_dim]
    return Z


#### Example of Attention At work

In [ ]:
multihead_attn = MultiHeadAttention(config)
attn_outputs_multihead = multihead_attn(input_embedding)
attn_outputs_multihead.size()

torch.Size([1, 5, 768])

In [ ]:
from bertviz import head_view
from transformers import AutoModel

model = AutoModel.from_pretrained(model_ckpt, output_attentions=True)

sent_a = "time flies like an arrow"
sent_b = "fruit flies like a banana"

viz_inputs = tokenizer(sent_a, sent_b, return_tensors='pt')


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
viz_inputs

{'input_ids': tensor([[  101,  2051, 10029,  2066,  2019,  8612,   102,  5909, 10029,  2066,
          1037, 15212,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [ ]:
viz_inputs.input_ids.shape

torch.Size([1, 13])

In [ ]:
tokenizer.decode(viz_inputs.input_ids[0])

'[CLS] time flies like an arrow [SEP] fruit flies like a banana [SEP]'

In [ ]:
model(**viz_inputs).keys()

odict_keys(['last_hidden_state', 'pooler_output', 'attentions'])

In [ ]:
model(**viz_inputs).attentions[0].shape #attention matrix---> [batch, heads, tokens, tokens]
# attention is calculated for each token for every other token

torch.Size([1, 12, 13, 13])

In [ ]:
attention = model(**viz_inputs).attentions
sent_b_start = (viz_inputs.token_type_ids == 0).sum(dim=1)
tokens = tokenizer.convert_ids_to_tokens(viz_inputs.input_ids[0])
head_view(attention, tokens, sent_b_start,heads=[8])

<IPython.core.display.Javascript object>

## Part 2: Feed-Foward Layer

In [ ]:
config

BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.20.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [ ]:
class FeedForward(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.linear1 = nn.Linear(config.hidden_size, config.intermediate_size)
    self.linear2 = nn.Linear(config.intermediate_size, config.hidden_size)
    self.gelu = nn.GELU()
    self.dropout = nn.Dropout(config.hidden_dropout_prob)

  def forward(self, attention_outputs):
    output_l1 = self.linear1(attention_outputs)
    activated_outputs = self.gelu(output_l1)
    output_l2 = self.linear2(activated_outputs)
    output = self.dropout(output_l2)
    return output




In [ ]:
from torch.nn.modules.activation import MultiheadAttention

# Step 1:
multihead_attn = MultiHeadAttention(config)
print(multihead_attn)
multi_head_attention_outputs = multihead_attn(input_embedding)
# Step 2
feed_forward_layer = FeedForward(config)
print(feed_forward_layer)
ff_outputs = feed_forward_layer(multi_head_attention_outputs)

MultiHeadAttention(
  (w_0): Linear(in_features=768, out_features=768, bias=True)
)
FeedForward(
  (linear1): Linear(in_features=768, out_features=3072, bias=True)
  (linear2): Linear(in_features=3072, out_features=768, bias=True)
  (gelu): GELU()
  (dropout): Dropout(p=0.1, inplace=False)
)


In [ ]:
ff_outputs.size()

torch.Size([1, 5, 768])

## Part 3: Adding Layer Normalization

#### Example: Understanding Layer Normalization

In [ ]:
layer_norm1 = nn.LayerNorm(config.hidden_size)

In [ ]:
l1 = layer_norm1(input_embedding)

In [ ]:
l1.shape

torch.Size([1, 5, 768])

In [ ]:
one_token = input_embedding[0][1,:].reshape(1,768)

In [ ]:
#l1 #sequence of 1 token embedding
#applying layer norm across all the features 768 for that token
layer_norm1 = nn.LayerNorm(config.hidden_size)
layer_norm_token = layer_norm1(one_token)
print(layer_norm_token.mean())
print(layer_norm_token.var())


tensor(-3.7253e-09, grad_fn=<MeanBackward0>)
tensor(1.0013, grad_fn=<VarBackward0>)


In [ ]:
batch, sentence_length, embedding_dim = 2, 3, 3
embedding = torch.randint(5, (2,3,3)).to(torch.float32)
print(embedding)
layer_norm = nn.LayerNorm(embedding_dim)
normalized = layer_norm(embedding)
print(normalized)

tensor([[[0., 2., 2.],
         [4., 1., 3.],
         [1., 3., 1.]],

        [[4., 2., 0.],
         [3., 0., 4.],
         [3., 2., 3.]]])
tensor([[[-1.4142,  0.7071,  0.7071],
         [ 1.0690, -1.3363,  0.2673],
         [-0.7071,  1.4142, -0.7071]],

        [[ 1.2247,  0.0000, -1.2247],
         [ 0.3922, -1.3728,  0.9806],
         [ 0.7071, -1.4142,  0.7071]]], grad_fn=<NativeLayerNormBackward0>)


In [ ]:
config

BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.20.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

## Part 4: Encoder Only Transformer Block

#### Single Transformer Layer

In [ ]:
class TransformerEncoderLayer(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.layer_norm1 = nn.LayerNorm(config.hidden_size)
    self.layer_norm2 = nn.LayerNorm(config.hidden_size)
    self.multi_attention = MultiHeadAttention(config)
    self.feedforward = FeedForward(config)

  def forward(self, input_embeddings):
     #pre-layer normalization approach

     #Step 1: Applying Layer Normalization to Input Embeddings
     normalized_input_embeddings = self.layer_norm1(input_embeddings)

     #Step 2: Applying MultiHeadAttention to Normalized Output
     multi_head_attn = self.multi_attention(normalized_input_embeddings)

     #Step 3: Add input embeddings to the Multihead Attention Output
     skip_connection_1 = input_embeddings + multi_head_attn

     #step 4: Pass the output to another Layer Normalization
     layer_norm_2 = self.layer_norm2(skip_connection_1)

     #Step 5: Adding skip connection 1 outputs to the output of the FeedForward Network (applied on Step 4)
     skip_connection_2 = skip_connection_1 + self.feedforward(layer_norm_2)
     return skip_connection_2


In [ ]:
encoder_only = TransformerEncoderLayer(config)
encoder_only

TransformerEncoderLayer(
  (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (multi_attention): MultiHeadAttention(
    (w_0): Linear(in_features=768, out_features=768, bias=True)
  )
  (feedforward): FeedForward(
    (linear1): Linear(in_features=768, out_features=3072, bias=True)
    (linear2): Linear(in_features=3072, out_features=768, bias=True)
    (gelu): GELU()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [ ]:
encoder_output = encoder_only(input_embedding)
encoder_output.size()

torch.Size([1, 5, 768])

#### Positional Encoding

In [ ]:
seq_length = input_embedding.size(1)
print(f'Sequence length {seq_length}')
position_ids = torch.arange(seq_length, dtype=torch.long).unsqueeze(0)
print(f'Position IDs for the Sequence {position_ids}')
#Embedding Layer Acts as a
position_embedding = nn.Embedding(config.max_position_embeddings, config.hidden_size)
embeddings = position_embedding(position_ids)
print(f'Position Embedding Shape {embeddings.shape}')

Sequence length 5
Position IDs for the Sequence tensor([[0, 1, 2, 3, 4]])
Position Embedding Shape torch.Size([1, 5, 768])


#### Embedding class
- Creates a single Dense Embedding for each token --> Token Embedding + Positional Embedding
- Input Shape is the number of token ids in a sequence ---> [batch_size,seq_len]
- Token and Positional Embedding Dense Representation for each token and output is [batch_size, seq_len, token_embeddings]

In [ ]:
class Embeddings(nn.Module):
  """
  Creates a single Dense Embedding for each token --> Token Embedding + Positional Embedding
  """
  def __init__(self,config):
    super().__init__()
    self.token_embedding = nn.Embedding(config.vocab_size, config.hidden_size)
    self.position_embedding = nn.Embedding(config.max_position_embeddings, config.hidden_size)
    self.layer_norm = nn.LayerNorm(config.hidden_size, eps= 1e-12)
    self.dropout = nn.Dropout()

  def forward(self,input_ids):
    token_embeddings = self.token_embedding(input_ids)
    seq_length = input_embedding.size(1)
    position_ids = torch.arange(seq_length, dtype=torch.long).unsqueeze(0)
    position_embeddings = self.position_embedding(position_ids)
    combined_embeddings = token_embeddings + position_embeddings
    normalized_embedding = self.layer_norm(combined_embeddings)
    normalized_embedding = self.dropout(normalized_embedding)
    return normalized_embedding



In [ ]:
embedding = Embeddings(config)
combined_embedding= embedding(inputs.input_ids)
print(f'Input Shape is the number of token ids in a sequence {inputs.input_ids.shape}')
print(f'Token and Positional Embedding Dense Representation for each token {combined_embedding.shape}')

Input Shape is the number of token ids in a sequence torch.Size([1, 5])
Token and Positional Embedding Dense Representation for each token torch.Size([1, 5, 768])


#### Creating TransferEncoder Class

In [ ]:
config

BertConfig {
  "_name_or_path": "bert-base-uncased",
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "classifier_dropout": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "transformers_version": "4.20.1",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}

In [ ]:
class TransferEncoder(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.embedding = Embeddings(config)
    self.layers = nn.ModuleList([TransformerEncoderLayer(config) for _ in range(config.num_hidden_layers)])

  def forward(self, input_ids):
    embeddings = self.embedding(input_ids)
    for layer in self.layers:
      embeddings = layer(embeddings)
    return embeddings




In [ ]:
encoder = TransferEncoder(config)
transformer_encoder_only_embeddings = encoder(inputs.input_ids)

In [ ]:
transformer_encoder_only_embeddings.size()

torch.Size([1, 5, 768])

## Part 5: Adding a Classification Head to the transformer Encoder Block

In [ ]:
class TransformerEncoderForTokenClassification(nn.Module):
  def __init__(self,config):
    super().__init__()
    self.encoder_embeddings = TransferEncoder(config)
    self.dropout = nn.Dropout(config.hidden_dropout_prob)
    self.classifier = nn.Linear(config.hidden_size, config.num_labels)

  def forward(self, input_ids):
    encoder_embeddings= self.encoder_embeddings(input_ids)[:,0,:] #Embedding of first token for each sequence  [CLS]
    drop = self.dropout(encoder_embeddings)
    classify = self.classifier(drop)
    return classify


In [ ]:
config.num_labels = 3
encoder_model = TransformerEncoderForTokenClassification(config)
classify_logits = encoder_model(inputs.input_ids)

In [ ]:
print(f'Input Shape Shape [Batch_size, seq_len] {classify_logits.shape}')
print(f'Prediction Shape [Batch_size, K (num_labels)] {classify_logits.shape}')

Input Shape Shape [Batch_size, seq_len] torch.Size([1, 3])
Prediction Shape [Batch_size, K (num_labels)] torch.Size([1, 3])
